In [ ]:
#参照：https://zenn.dev/datajournal1/articles/37d30a8b54769a
import cv2
import os
import subprocess
import re
from datetime import datetime

#自分が接続しているカメラの型番をここに入れる
def get_camera_id(cameara_name_keyword="c270"):
    try:
        #デバイス検索結果の返却
        result = subprocess.check_output(['v4l2-cli', '--list-devices'], text=True)
        #デバイス名とパスのペアを抽出する
        parts = result.split('\n\n')
        for part in parts:
            if camera_name_keyword in part:
                match = re.search(r'/dev/video(\d+)', part)
                if match:
                    found_id = int(match.group(1))
                    print(f"Found {camera_name_keyword}'s pid is /dev/video{found_id}")
                    return found_id
    except Exception as e:
        print(f"ERROR SEARCHING FOR CAMERA : {e}")

    print(f"Camera : {camera_name_keyword} is not found - > return ID = 0")
    return 0
    
def capture_images(cam_id, output_dir, num_images=100):
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(cam_id, cv2.CAP_V4L2)
    if not cap.isOpened():
        print("カメラの使用不可状態 -> pid間違いやポートの割り当てを確認してください")
        return
        
    count = 0
    print(f"スペースキーで撮影、qキーで中断(終了枚数：{num_images})")
    while count < num_images:
        ret, frame = cap.read()
        if not ret:
            print("フレームの取得失敗")
            break
        cv2.imshow('Camera Preview', frame)

        key = cv2.waitkey(1) & 0xFF
        if key == ord(' '):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            file_name = f"{output_dir}/img_{timestamp}.jpg"
            cv2.imwrite(file_name, frame)
            count += 1
            print(f"Captured {count}/{num_images}")
        elif key == ord('q'):
            print("Check keyboard intrrupt -> 終了")
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    cam_id = get_camera_id()
    capture_images(cam_id, "data", num_images=500)